# Notebook 5 — Statistical analysis, fusion, bias and final tables (CPU only)

Uses only the saved predictions; no model is loaded.
1. Paired significance tests (question-level cluster bootstrap, Holm-corrected)
2. Lexical + LLM fusion: original rule vs weights learned on validation data
3. Error analysis: errors by gold-score band, and answer-length bias
4. SciEntsBank paired tests on macro-F1 and weighted-F1 (5-way macro-F1 is dominated by the tiny non-domain class: 3 / 0 / 20 test answers)
5. Final formatted tables for the manuscript

Can be re-run at any time; the SciEntsBank section is skipped until its fine-tuned predictions exist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json
ROOT = '/content/drive/MyDrive/asag_revision'
sys.path.insert(0, ROOT)
import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from asag_utils import *
RES = f'{ROOT}/results'
mo = pd.read_csv(f'{ROOT}/data/mohler_prepared.csv')
preds = pd.read_csv(f'{RES}/mohler_preds.csv')
assert 'llama3_ft' in preds, 'Run Notebook 3 to completion first'

## 0. Paired bootstrap helper

In [ ]:
N_BOOT = 2000

def paired_bootstrap(y, a, b, metric_fn, groups=None, n_boot=N_BOOT, seed=SEED):
    """Difference metric(a) - metric(b) on the SAME resampled answers (whole questions if groups given).
    Returns {metric: (diff, ci_lo, ci_hi, p_two_sided)}."""
    y, a, b = np.asarray(y), np.asarray(a), np.asarray(b)
    rng = np.random.default_rng(seed)
    if groups is not None:
        groups = np.asarray(groups)
        sets = [np.flatnonzero(groups == g) for g in pd.unique(groups)]
    ma, mb = metric_fn(y, a), metric_fn(y, b)
    diffs = []
    for _ in range(n_boot):
        idx = (rng.integers(0, len(y), len(y)) if groups is None
               else np.concatenate([sets[i] for i in rng.integers(0, len(sets), len(sets))]))
        A, B = metric_fn(y[idx], a[idx]), metric_fn(y[idx], b[idx])
        diffs.append({k: A[k] - B[k] for k in A})
    D = pd.DataFrame(diffs)
    out = {}
    for k in ma:
        d = D[k].dropna().values
        p = max(2 * min((d <= 0).mean(), (d >= 0).mean()), 1 / n_boot)
        out[k] = (ma[k] - mb[k], np.percentile(d, 2.5), np.percentile(d, 97.5), min(p, 1.0))
    return out

def holm(p):
    """Holm-Bonferroni adjusted p-values."""
    p = np.asarray(p, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0
    for r, i in enumerate(order):
        run = max(run, (m - r) * p[i]); adj[i] = min(run, 1.0)
    return adj

## 1. Fusion of the lexical composite with the fine-tuned LLM (per fold; weights fitted on validation answers only)

In [ ]:
feats = pd.read_csv(f'{ROOT}/cache/features.csv')
ft = pd.concat([pd.read_csv(f'{RES}/mohler_llama3_ft_f{k}.csv') for k in range(N_FOLDS)])
COMP = ['context', 'keyword', 'synonym', 'grammar']
M = mo.merge(feats, on='uid')
fused, fw = [], []
for k in range(N_FOLDS):
    s = M[f'split_f{k}']
    tr, va, te = M[s == 'train'], M[s == 'val'].copy(), M[s == 'test'].copy()
    lex = LinearRegression(positive=True).fit(tr[COMP], tr.y)          # lexical model: train answers only
    fk = ft[ft.fold == k].set_index('uid').expected
    for d in (va, te):
        d['lex'] = np.clip(lex.predict(d[COMP]), 0, 10)
        d['llm'] = d.uid.map(fk).values
    assert va.llm.notna().all() and te.llm.notna().all()
    fus = LinearRegression(positive=True).fit(va[['lex', 'llm']], va.y)  # fusion weights: validation answers only
    w = fus.coef_ / fus.coef_.sum() if fus.coef_.sum() > 0 else fus.coef_
    fw.append({'fold': k, 'w_lexical': w[0], 'w_llm': w[1], 'intercept': fus.intercept_})
    fused.append(pd.DataFrame({
        'uid': te.uid,
        'fusion_original': 0.3 * te.composite_fixed + 0.7 * te.llm,          # original paper rule (raw composite)
        'fusion_fixed_30_70': 0.3 * te.lex + 0.7 * te.llm,                   # same weights, lexical score on the right scale
        'fusion_learned': np.clip(fus.predict(te[['lex', 'llm']]), 0, 10)}))
fw = pd.DataFrame(fw); fw.to_csv(f'{RES}/nb5_fusion_weights.csv', index=False)
preds = preds.drop(columns=[c for c in ['fusion_original', 'fusion_fixed_30_70', 'fusion_learned'] if c in preds]) \
             .merge(pd.concat(fused), on='uid')
preds.to_csv(f'{RES}/mohler_preds.csv', index=False)
display(fw.round(3))
rows = [{'system': s, **summarise(preds.y, preds[s], reg_metrics, groups=preds.group, folds=preds.fold)}
        for s in ['llama3_ft', 'fusion_original', 'fusion_fixed_30_70', 'fusion_learned']]
fusion_summary = pd.DataFrame(rows); fusion_summary.to_csv(f'{RES}/mohler_nb5_fusion_summary.csv', index=False)
fusion_summary[['system', 'pearson', 'pearson_ci', 'mae', 'rmse', 'qwk', 'pearson_fold_std']].round(3)

## 2. Mohler: paired comparisons (cluster bootstrap over questions, Holm correction on Pearson)

In [ ]:
COMPARISONS = [
    ('llama3_ft', 'llama3_base_zs_cal', 'Fine-tuning vs the same base model zero-shot'),
    ('llama3_ft', 'llama3_instruct_zs', 'Fine-tuned vs Instruct zero-shot'),
    ('llama3_ft', 'llama3_instruct_zs_cal', 'Fine-tuned vs Instruct zero-shot (rescaled)'),
    ('llama3_ft', 'composite_fixed_cal', 'Fine-tuned vs lexical composite'),
    ('llama3_instruct_zs', 'composite_fixed_cal', 'Instruct zero-shot vs lexical composite'),
    ('fusion_learned', 'llama3_ft', 'Learned fusion vs fine-tuned alone'),
    ('fusion_original', 'llama3_ft', 'Original 30/70 fusion vs fine-tuned alone'),
]
rows = []
for a, b, desc in COMPARISONS:
    r = paired_bootstrap(preds.y, preds[a], preds[b], reg_metrics, groups=preds.group)
    row = {'A': a, 'B': b, 'comparison': desc}
    for m in ['pearson', 'mae', 'qwk']:
        d, lo, hi, p = r[m]
        row.update({f'd_{m}': d, f'd_{m}_ci': f'[{lo:.3f}, {hi:.3f}]', f'p_{m}': p})
    rows.append(row)
mo_tests = pd.DataFrame(rows)
mo_tests['p_pearson_holm'] = holm(mo_tests.p_pearson)
mo_tests.to_csv(f'{RES}/mohler_paired_tests.csv', index=False)
mo_tests[['comparison', 'd_pearson', 'd_pearson_ci', 'p_pearson_holm', 'd_mae', 'd_mae_ci', 'd_qwk', 'd_qwk_ci']].round(4)

Positive `d_pearson`/`d_qwk` and negative `d_mae` favour system A.

## 3. Error analysis: by gold-score band, and answer-length bias

In [ ]:
SYS = ['composite_fixed_cal', 'llama3_instruct_zs_cal', 'llama3_ft', 'fusion_learned']
E = preds.merge(mo[['uid', 'student_answer']], on='uid')
E['length'] = E.student_answer.astype(str).str.split().str.len()
E['band'] = pd.cut(E.y, [-0.01, 5.99, 8.99, 10], labels=['low (<6)', 'mid (6-8.75)', 'high (>=9)'])
band = []
for s in SYS:
    for b, d in E.groupby('band', observed=True):
        band.append({'system': s, 'band': b, 'n': len(d), 'mae': np.abs(d[s] - d.y).mean(),
                     'mean_signed_error': (d[s] - d.y).mean()})
band = pd.DataFrame(band); band.to_csv(f'{RES}/mohler_error_by_band.csv', index=False)
BANDS = ['low (<6)', 'mid (6-8.75)', 'high (>=9)']
pv = band.pivot_table(index='system', columns='band', values=['mae', 'mean_signed_error'], observed=True)
display(pv.reindex(columns=pd.MultiIndex.from_product([['mae', 'mean_signed_error'], BANDS])).round(3))
print('Answers per band:', E.band.value_counts().reindex(BANDS).to_dict())

E['len_tercile'] = pd.qcut(E.length, 3, labels=['short', 'medium', 'long'])
lb = []
for s in SYS:
    res = E[s] - E.y
    rho = pd.Series(res).corr(E.length, method='spearman')
    row = {'system': s, 'spearman_residual_vs_length': rho}
    for t, d in E.groupby('len_tercile', observed=True):
        row[f'signed_err_{t}'] = (d[s] - d.y).mean()
    lb.append(row)
lb = pd.DataFrame(lb); lb.to_csv(f'{RES}/mohler_length_bias.csv', index=False)
print('Length terciles (words):', E.groupby('len_tercile', observed=True).length.agg(['min', 'max']).to_dict('index'))
lb.round(3)

`mean_signed_error` > 0 means the system over-scores. A positive residual–length correlation means longer answers are over-scored.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(SYS), figsize=(4 * len(SYS), 3.8), sharey=True)
for a, s in zip(ax, SYS):
    a.scatter(E.y + np.random.default_rng(0).uniform(-.1, .1, len(E)), E[s], s=4, alpha=.3)
    a.plot([0, 10], [0, 10], 'k--', lw=1); a.set_title(s, fontsize=9); a.set_xlabel('gold score')
ax[0].set_ylabel('predicted score')
plt.tight_layout(); plt.savefig(f'{RES}/fig_mohler_pred_vs_gold.png', dpi=300); plt.show()

## 4. SciEntsBank paired comparisons (skipped until Notebook 4 is finished)

In [ ]:
sp = pd.read_csv(f'{RES}/scientsbank_preds.csv')
label_col = {'5way': 'label', '3way': 'label3', '2way': 'label2'}
se_tests = None
if 'llama3_ft_5way' not in sp:
    print('SciEntsBank fine-tuned predictions not found yet: run Notebook 4, then re-run this notebook.')
else:
    SE_COMP = [('llama3_ft', 'llama3_base_zs_cal', 'Fine-tuning vs same base zero-shot (calibrated)'),
               ('llama3_ft', 'llama3_instruct_zs_cal', 'Fine-tuned vs Instruct zero-shot (calibrated)'),
               ('llama3_ft', 'lexical_4', 'Fine-tuned vs lexical features')]
    rows = []
    for scheme, col in label_col.items():
        for a, b, desc in SE_COMP:
            for t in ['test_ua', 'test_uq', 'test_ud']:
                d = sp[(sp.split == t)].dropna(subset=[f'{a}_{scheme}', f'{b}_{scheme}'])
                r = paired_bootstrap(d[col], d[f'{a}_{scheme}'], d[f'{b}_{scheme}'], cls_metrics,
                                     groups=None if t == 'test_ua' else d.group)
                dd, lo, hi, p = r['macro_f1']
                dw, wlo, whi, pw = r['weighted_f1']
                rows.append({'scheme': scheme, 'comparison': desc, 'test': t, 'd_macro_f1': dd,
                             'ci': f'[{lo:.3f}, {hi:.3f}]', 'p': p,
                             'd_weighted_f1': dw, 'ci_weighted': f'[{wlo:.3f}, {whi:.3f}]', 'p_weighted': pw})
    se_tests = pd.DataFrame(rows)
    se_tests['p_holm'] = np.nan
    for scheme in label_col:                       # Holm within each labelling scheme
        m = se_tests.scheme == scheme
        se_tests.loc[m, 'p_holm'] = holm(se_tests.loc[m, 'p'])
        se_tests.loc[m, 'p_weighted_holm'] = holm(se_tests.loc[m, 'p_weighted'])
    se_tests.to_csv(f'{RES}/scientsbank_paired_tests.csv', index=False)
    display(se_tests.round(4))

## 5. Final tables for the manuscript

In [ ]:
def fmt(v, ci=None):
    return f'{v:.3f}' if ci is None or pd.isna(ci) else f'{v:.3f} {ci}'

mo_all = pd.concat([pd.read_csv(f'{RES}/mohler_nb0_summary.csv'), pd.read_csv(f'{RES}/mohler_nb1_summary.csv'),
                    pd.read_csv(f'{RES}/mohler_nb3_summary.csv'), fusion_summary])
MO_ROWS = {'mean': 'Mean predictor', 'tfidf_cos': 'TF-IDF cosine', 'minilm_cos': 'MiniLM cosine',
           'composite_fixed': 'Lexical composite, original weights (raw)',
           'composite_fixed_cal': 'Lexical composite, original weights (rescaled)',
           'composite_learned': 'Lexical composite, learned weights',
           'llama3_base_zs_cal': 'Llama 3 8B zero-shot (rescaled)',
           'llama3_instruct_zs': 'Llama 3 8B Instruct zero-shot',
           'llama3_instruct_zs_cal': 'Llama 3 8B Instruct zero-shot (rescaled)',
           'llama3_ft': 'Llama 3 8B fine-tuned (LoRA)', 'llama3_ft_cal': 'Llama 3 8B fine-tuned (rescaled)',
           'fusion_original': 'Fusion, original 30/70 rule', 'fusion_learned': 'Fusion, learned weights',
           'human_grader1_vs_grader2': 'Human grader 1 vs grader 2 (reference)'}
t = mo_all.drop_duplicates('system', keep='last').set_index('system').loc[list(MO_ROWS)]
table_mo = pd.DataFrame({'System': list(MO_ROWS.values()),
                         'Pearson [95% CI]': [fmt(r.pearson, r.get('pearson_ci')) if pd.notna(r.pearson) else '–' for _, r in t.iterrows()],
                         'Spearman': [f'{v:.3f}' if pd.notna(v) else '–' for v in t.spearman],
                         'MAE': t.mae.round(3).values, 'RMSE': t.rmse.round(3).values,
                         'QWK': [f'{v:.3f}' if pd.notna(v) else '–' for v in t.qwk],
                         'Pearson across folds': [f'{m:.3f} ± {s:.3f}' if pd.notna(m) else '–'
                                                  for m, s in zip(t.get('pearson_fold_mean'), t.get('pearson_fold_std'))]})
table_mo.to_csv(f'{RES}/TABLE_mohler_main.csv', index=False)

abl = mo_all.drop_duplicates('system', keep='last').set_index('system')
base = abl.loc['composite_learned', 'pearson']
table_abl = pd.DataFrame([{'Removed component': c, 'Pearson': abl.loc[f'composite_learned_minus_{c}', 'pearson'],
                           'Δ vs full composite': abl.loc[f'composite_learned_minus_{c}', 'pearson'] - base,
                           'Learned weight (mean over folds)': pd.read_csv(f'{RES}/nb0_weights.csv')[c].mean()}
                          for c in COMP]).round(3)
table_abl.to_csv(f'{RES}/TABLE_mohler_lexical_ablation.csv', index=False)
display(table_mo, table_abl)

In [ ]:
if 'llama3_ft_5way' in sp:
    se_all = pd.concat([pd.read_csv(f'{RES}/scientsbank_nb0_summary.csv'), pd.read_csv(f'{RES}/scientsbank_nb2_summary.csv'),
                        pd.read_csv(f'{RES}/scientsbank_nb4_summary.csv')])
    SE_ROWS = {'majority': 'Majority class', 'lexical_4': 'Lexical features (logistic regression)',
               'llama3_base_zs_cal': 'Llama 3 8B zero-shot (calibrated)',
               'llama3_instruct_zs': 'Llama 3 8B Instruct zero-shot',
               'llama3_instruct_zs_cal': 'Llama 3 8B Instruct zero-shot (calibrated)',
               'llama3_ft': 'Llama 3 8B fine-tuned (LoRA)', 'llama3_ft_cal': 'Llama 3 8B fine-tuned (calibrated)'}
    out = []
    for key, name in SE_ROWS.items():
        row = {'System': name}
        for scheme in ['5way', '3way', '2way']:
            for tst, lab in [('test_ua', 'UA'), ('test_uq', 'UQ'), ('test_ud', 'UD')]:
                r = se_all[(se_all.system == key) & (se_all.scheme == scheme) & (se_all.test == tst)].iloc[-1]
                row[f'{scheme} {lab} macro-F1'] = f'{r.macro_f1:.3f}'
                row[f'{scheme} {lab} weighted-F1'] = f'{r.weighted_f1:.3f}'
                row[f'{scheme} {lab} acc'] = f'{r.acc:.3f}'
        out.append(row)
    table_se = pd.DataFrame(out); table_se.to_csv(f'{RES}/TABLE_scientsbank_main.csv', index=False)
    display(table_se)